---
## Trabajo Autónomo

Para consolidar esta sesión, cada grupo deberá elaborar la **Ficha del Dataset del Proyecto Integrador** y subirla a la plataforma de Classroom.

Asegúrese de incluir en su informe:
1. La delimitación de la pregunta de negocio y la variable objetivo elegida.
2. La fuente del dato, el responsable de su gestión y la autorización de uso.
3. El diccionario de datos detallando tipos, unidades físicas y límites lógicos de cada sensor.
4. La identificación de riesgos de calidad latentes (ej. centinelas de error de hardware) y su tratamiento justificado.
5. El enlace al Notebook formativo 2 reproducible y debidamente comentado por los participantes.

---

# Aplicación al Proyecto Integrador
## AndinaLog Subcaso 03B Operaciones y Planificación de Datos

Esta sección aplica el mismo proceso de ingesta, inspección y diagnóstico a los archivos Bronze asignados al Grupo 6. Los datos originales se conservan en objetos con el sufijo `_raw`; las transformaciones se realizan sobre copias para mantener la trazabilidad.

El propósito es respaldar la **Ficha del Dataset del Proyecto Integrador** con resultados reproducibles de `df.info()`, tipos, nulos, duplicados y alertas de calidad.

## 7. Preparación de la carpeta en Google Drive

1. En Google Drive, cree la carpeta `Mi unidad/GIAD/AndinaLog_03B`.
2. Suba dentro de esa carpeta los nueve archivos del subcaso `GIAD_M3_Subcaso_03B_Bronze`.
3. Mantenga los nombres originales de los archivos.
4. Ejecute la siguiente celda y autorice a Colab para leer su Drive.

La ruta que Colab verá será `/content/drive/MyDrive/GIAD/AndinaLog_03B`. Si utiliza otra carpeta, cambie únicamente la variable `BASE`.

In [26]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/GIAD/AndinaLog_03B")

ARCHIVOS_ESPERADOS = [
    "andinalog_wms_orders.csv",
    "andinalog_flota.csv",
    "andinalog_iot_telemetry.csv",
    "andinalog_warehouse_costs.csv",
    "andinalog_hr_drivers.csv",
    "andinalog_productos.csv",
    "andinalog_inventory_tracking.csv",
    "andinalog_flota_eventos.json",
    "andinalog_bitacora_choferes.txt",
]

faltantes = [nombre for nombre in ARCHIVOS_ESPERADOS if not (BASE / nombre).exists()]

if faltantes:
    raise FileNotFoundError(
        "No se encontraron estos archivos en Drive:\n- " + "\n- ".join(faltantes)
        + f"\n\nRevise la ruta configurada: {BASE}"
    )

print(f"Carpeta encontrada: {BASE}")
print(f"Archivos verificados: {len(ARCHIVOS_ESPERADOS)} de {len(ARCHIVOS_ESPERADOS)}")

Carpeta encontrada: /content/drive/MyDrive/GIAD/AndinaLog_03B
Archivos verificados: 9 de 9


## 8. Ingesta de los archivos Bronze

Los CSV se cargan sin modificar su fuente. Se declaran de forma explícita los valores que representan ausencia. El JSON se aplana a una tabla de eventos y la bitácora TXT se estructura solo cuando cada línea contiene los cuatro campos esperados.

In [28]:
import json
import numpy as np
import pandas as pd
from IPython.display import display

print("Versión de pandas:", pd.__version__)

NA_VALUES = ["", "NA", "N/A", "NULL", "—"]

ARCHIVOS_CSV = {
    "Pedidos WMS": "andinalog_wms_orders.csv",
    "Flota": "andinalog_flota.csv",
    "Telemetría IoT": "andinalog_iot_telemetry.csv",
    "Costos de almacén": "andinalog_warehouse_costs.csv",
    "Choferes": "andinalog_hr_drivers.csv",
    "Productos": "andinalog_productos.csv",
    "Inventario": "andinalog_inventory_tracking.csv",
}

datos_raw = {
    nombre: pd.read_csv(
        BASE / archivo,
        sep=",",
        na_values=NA_VALUES,
        keep_default_na=True,
        encoding="utf-8",
    )
    for nombre, archivo in ARCHIVOS_CSV.items()
}

with open(BASE / "andinalog_flota_eventos.json", encoding="utf-8") as archivo:
    flota_json = json.load(archivo)

eventos = []
for camion in flota_json.get("camiones", []):
    for evento in camion.get("eventos", []):
        eventos.append({"camion_id": camion.get("camion_id"), **evento})
df_eventos_raw = pd.DataFrame(eventos)

lineas_bitacora_raw = [
    linea for linea in (BASE / "andinalog_bitacora_choferes.txt")
    .read_text(encoding="utf-8")
    .splitlines()
    if linea.strip()
]

registros_bitacora = []
bitacora_malformadas = []
for numero_linea, linea in enumerate(lineas_bitacora_raw, start=1):
    partes = [parte.strip() for parte in linea.split("|")]
    if len(partes) == 4:
        registros_bitacora.append(partes)
    else:
        bitacora_malformadas.append({"linea": numero_linea, "contenido": linea})

df_bitacora_raw = pd.DataFrame(
    registros_bitacora,
    columns=["fecha", "turno", "camion_id", "observacion"],
).replace("", pd.NA)

datos_todos_raw = {
    **datos_raw,
    "Eventos de flota JSON": df_eventos_raw,
    "Bitácora de choferes TXT": df_bitacora_raw,
}

print("Fuentes cargadas:", len(datos_todos_raw))
print("Líneas de bitácora que requieren revisión manual:", len(bitacora_malformadas))

Versión de pandas: 2.2.3
Fuentes cargadas: 9
Líneas de bitácora que requieren revisión manual: 4


## 9. Radiografía técnica

Primero se inspecciona la estructura de cada fuente. No se limpian ni eliminan registros antes de registrar su estado original.

In [29]:
for nombre, dataframe in datos_todos_raw.items():
    print("\n" + "=" * 80)
    print(nombre)
    print("=" * 80)
    dataframe.info()
    print("\nTipos inferidos:")
    print(dataframe.dtypes)
    print("\nValores faltantes:")
    print(dataframe.isna().sum()[dataframe.isna().sum() > 0])


Pedidos WMS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7550 entries, 0 to 7549
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   order_id                      7550 non-null   object 
 1   cliente_id                    7550 non-null   object 
 2   producto_id                   7550 non-null   object 
 3   fecha_despacho                7550 non-null   object 
 4   centro_distribucion           7550 non-null   object 
 5   camion_id                     7550 non-null   object 
 6   chofer_id                     7550 non-null   object 
 7   cantidad_solicitada           7550 non-null   object 
 8   cantidad_entregada            7470 non-null   float64
 9   tiempo_entrega_prometido_hrs  7550 non-null   int64  
 10  tiempo_entrega_real_hrs       7550 non-null   float64
 11  otif_on_time                  7550 non-null   int64  
 12  otif_in_full                  7550 non-null   int

In [30]:
CLAVES_NEGOCIO = {
    "Pedidos WMS": ["order_id"],
    "Flota": ["camion_id"],
    "Telemetría IoT": ["timestamp", "viaje_id", "camion_id"],
    "Costos de almacén": ["centro_distribucion", "periodo_mes"],
    "Choferes": ["chofer_id"],
    "Productos": ["producto_id"],
    "Inventario": ["movimiento_id"],
    "Eventos de flota JSON": ["evento_id"],
}

def normalizar_texto(serie):
    return serie.astype("string").str.strip().str.upper()

def filas_en_claves_duplicadas(dataframe, claves):
    claves_normalizadas = pd.DataFrame({
        columna: normalizar_texto(dataframe[columna])
        for columna in claves
    })
    return int(claves_normalizadas.duplicated(keep=False).sum())

filas_resumen = []
for nombre, dataframe in datos_todos_raw.items():
    claves = CLAVES_NEGOCIO.get(nombre)
    filas_resumen.append({
        "fuente": nombre,
        "filas": len(dataframe),
        "columnas": dataframe.shape[1],
        "nulos_totales": int(dataframe.isna().sum().sum()),
        "duplicados_exactos": int(dataframe.duplicated().sum()),
        "filas_en_claves_duplicadas": (
            filas_en_claves_duplicadas(dataframe, claves) if claves else pd.NA
        ),
    })

resumen_auditoria = pd.DataFrame(filas_resumen)
display(resumen_auditoria)

,fuente,filas,columnas,nulos_totales,duplicados_exactos,filas_en_claves_duplicadas
0,Pedidos WMS,7550,14,80,49,100
1,Flota,32,4,0,2,4
2,Telemetría IoT,28920,10,180,114,240
3,Costos de almacén,6,5,1,0,2
4,Choferes,156,6,0,4,10
5,Productos,63,7,2,2,6
6,Inventario,6040,12,18,39,80
7,Eventos de flota JSON,196,7,21,11,22
8,Bitácora de choferes TXT,143,4,24,6,<NA>


In [31]:
detalle_nulos = (
    pd.concat(
        {
            nombre: dataframe.isna().sum()
            for nombre, dataframe in datos_todos_raw.items()
        },
        names=["fuente", "variable"],
    )
    .rename("cantidad_nulos")
    .reset_index()
)

detalle_nulos = detalle_nulos[detalle_nulos["cantidad_nulos"] > 0]
detalle_nulos["porcentaje"] = detalle_nulos.apply(
    lambda fila: round(
        fila["cantidad_nulos"] / len(datos_todos_raw[fila["fuente"]]) * 100,
        2,
    ),
    axis=1,
)

display(detalle_nulos.sort_values(["fuente", "cantidad_nulos"], ascending=[True, False]))

,fuente,variable,cantidad_nulos,porcentaje
66,Bitácora de choferes TXT,turno,24,16.78
29,Costos de almacén,rotacion_stock_dias,1,16.67
64,Eventos de flota JSON,reconocido,21,10.71
52,Inventario,fecha_vencimiento,18,0.30
8,Pedidos WMS,cantidad_entregada,80,1.06
42,Productos,temperatura_conservacion_requerida_c,2,3.17
25,Telemetría IoT,humedad_cabina_pct,100,0.35
23,Telemetría IoT,temperatura_cabina_c,80,0.28


## 10. Alertas de calidad reproducibles

Las siguientes comprobaciones cuantifican los problemas que se documentan en la ficha. Un valor no convertible se marca como faltante para el diagnóstico, pero no se elimina de la fuente original.

In [32]:
def a_numero(serie):
    texto = serie.astype("string").str.strip().str.replace(",", ".", regex=False)
    return pd.to_numeric(texto, errors="coerce")

def a_fecha(serie):
    return pd.to_datetime(
        serie.astype("string").str.strip(),
        format="mixed",
        dayfirst=True,
        errors="coerce",
    )

def no_convertibles(dataframe, columna, conversion):
    original_presente = dataframe[columna].notna()
    convertido = conversion(dataframe[columna])
    return int((original_presente & convertido.isna()).sum())

alertas = []

def registrar(fuente, alerta, cantidad):
    alertas.append({"fuente": fuente, "alerta": alerta, "cantidad": int(cantidad)})

wms_raw = datos_raw["Pedidos WMS"]
registrar("Pedidos WMS", "cantidad_entregada faltante", wms_raw["cantidad_entregada"].isna().sum())
registrar("Pedidos WMS", "cantidad_solicitada no convertible", no_convertibles(wms_raw, "cantidad_solicitada", a_numero))
registrar("Pedidos WMS", "fecha_despacho no convertible", no_convertibles(wms_raw, "fecha_despacho", a_fecha))
registrar("Pedidos WMS", "tiempo_entrega_real_hrs negativo", (a_numero(wms_raw["tiempo_entrega_real_hrs"]) < 0).sum())
registrar("Pedidos WMS", "filas duplicadas exactas", wms_raw.duplicated().sum())
registrar("Pedidos WMS", "filas en order_id duplicado normalizado", filas_en_claves_duplicadas(wms_raw, ["order_id"]))

iot_raw = datos_raw["Telemetría IoT"]
temp_iot = a_numero(iot_raw["temperatura_cabina_c"])
humedad_iot = a_numero(iot_raw["humedad_cabina_pct"])
registrar("Telemetría IoT", "temperatura faltante", iot_raw["temperatura_cabina_c"].isna().sum())
registrar("Telemetría IoT", "humedad faltante", iot_raw["humedad_cabina_pct"].isna().sum())
registrar("Telemetría IoT", "código centinela -999", temp_iot.eq(-999).sum())
registrar("Telemetría IoT", "timestamp no convertible", no_convertibles(iot_raw, "timestamp", a_fecha))
registrar("Telemetría IoT", "humedad fuera de 0 a 100", ((humedad_iot < 0) | (humedad_iot > 100)).sum())
registrar("Telemetría IoT", "filas duplicadas exactas", iot_raw.duplicated().sum())
registrar("Telemetría IoT", "filas en clave de lectura duplicada", filas_en_claves_duplicadas(iot_raw, ["timestamp", "viaje_id", "camion_id"]))

inventario_raw = datos_raw["Inventario"]
fecha_ingreso = a_fecha(inventario_raw["fecha_ingreso"])
fecha_salida = a_fecha(inventario_raw["fecha_salida"])
fecha_vencimiento = a_fecha(inventario_raw["fecha_vencimiento"])
registrar("Inventario", "fecha_vencimiento faltante o no convertible", fecha_vencimiento.isna().sum())
registrar("Inventario", "cantidad_ingreso no convertible", no_convertibles(inventario_raw, "cantidad_ingreso", a_numero))
registrar("Inventario", "cantidad_merma negativa", (a_numero(inventario_raw["cantidad_merma"]) < 0).sum())
registrar("Inventario", "fecha_salida presente pero no convertible", (inventario_raw["fecha_salida"].notna() & fecha_salida.isna()).sum())
registrar("Inventario", "salida anterior al ingreso", ((fecha_salida < fecha_ingreso) & fecha_salida.notna()).sum())
registrar("Inventario", "vencimiento anterior al ingreso", ((fecha_vencimiento < fecha_ingreso) & fecha_vencimiento.notna()).sum())
registrar("Inventario", "filas duplicadas exactas", inventario_raw.duplicated().sum())
registrar("Inventario", "filas en movimiento_id duplicado normalizado", filas_en_claves_duplicadas(inventario_raw, ["movimiento_id"]))

productos_raw = datos_raw["Productos"]
registrar("Productos", "temperatura requerida faltante", productos_raw["temperatura_conservacion_requerida_c"].isna().sum())
registrar("Productos", "categoría escrita como conjelado", normalizar_texto(productos_raw["categoria_logistica"]).eq("CONJELADO").sum())
registrar("Productos", "filas duplicadas exactas", productos_raw.duplicated().sum())

flota_raw = datos_raw["Flota"]
registrar("Flota", "filas duplicadas exactas", flota_raw.duplicated().sum())
registrar("Flota", "filas en camion_id duplicado normalizado", filas_en_claves_duplicadas(flota_raw, ["camion_id"]))

costos_raw = datos_raw["Costos de almacén"]
registrar("Costos de almacén", "rotacion_stock_dias faltante", costos_raw["rotacion_stock_dias"].isna().sum())
registrar("Costos de almacén", "filas en centro y periodo duplicados", filas_en_claves_duplicadas(costos_raw, ["centro_distribucion", "periodo_mes"]))

registrar("Eventos JSON", "filas duplicadas exactas", df_eventos_raw.duplicated().sum())
registrar("Eventos JSON", "filas en evento_id duplicado", filas_en_claves_duplicadas(df_eventos_raw, ["evento_id"]))
registrar("Eventos JSON", "reconocido faltante", df_eventos_raw["reconocido"].isna().sum())

registrar("Bitácora TXT", "líneas duplicadas exactas", len(lineas_bitacora_raw) - len(set(lineas_bitacora_raw)))
registrar("Bitácora TXT", "turno omitido", sum("|  |" in linea or "| |" in linea for linea in lineas_bitacora_raw))
registrar("Bitácora TXT", "líneas con cantidad de campos incorrecta", len(bitacora_malformadas))

resumen_alertas = pd.DataFrame(alertas)
display(resumen_alertas)

,fuente,alerta,cantidad
0,Pedidos WMS,cantidad_entregada faltante,80
1,Pedidos WMS,cantidad_solicitada no convertible,15
2,Pedidos WMS,fecha_despacho no convertible,4
3,Pedidos WMS,tiempo_entrega_real_hrs negativo,15
4,Pedidos WMS,filas duplicadas exactas,49
5,Pedidos WMS,filas en order_id duplicado normalizado,100
6,Telemetría IoT,temperatura faltante,80
7,Telemetría IoT,humedad faltante,100
8,Telemetría IoT,código centinela -999,120
9,Telemetría IoT,timestamp no convertible,15


In [33]:
print("Unidades de temperatura encontradas:")
print(iot_raw["temp_unit"].astype("string").str.strip().str.upper().value_counts(dropna=False))

print("\nCategorías de producto encontradas:")
print(productos_raw["categoria_logistica"].value_counts(dropna=False))

print("\nEjemplos de fechas WMS que no se pueden convertir:")
display(wms_raw.loc[a_fecha(wms_raw["fecha_despacho"]).isna(), ["order_id", "fecha_despacho"]].head())

Unidades de temperatura encontradas:
temp_unit
C    28865
F       50
K        5
Name: count, dtype: Int64

Categorías de producto encontradas:
categoria_logistica
Seco         35
Fresco       16
Congelado     9
conjelado     3
Name: count, dtype: int64

Ejemplos de fechas WMS que no se pueden convertir:


,order_id,fecha_despacho
2783,ORD-2026-02784,2026-02-31 08:00:00
3081,ORD-2026-03082,2026-02-31 08:00:00
5119,ORD-2026-05120,2026-02-31 08:00:00
7323,ORD-2026-07324,2026-02-31 08:00:00


## 11. Diccionario de variables críticas

Los rangos siguientes son reglas de validación para el análisis. Las temperaturas de cabina deben compararse con el rango específico de cada producto, no con un único límite general.

In [34]:
diccionario_variables = pd.DataFrame([
    ["order_id, producto_id, camion_id, lote_id", "string / clave", "Sin unidad", "No nulo y formato normalizado; único según la granularidad"],
    ["fecha_despacho, timestamp", "datetime", "Fecha y hora", "Fecha válida y coherente con la secuencia operativa"],
    ["fecha_ingreso, fecha_salida, fecha_vencimiento", "date/datetime", "Fecha", "salida ≥ ingreso y vencimiento ≥ ingreso"],
    ["cantidades solicitada, entregada, ingreso, salida y merma", "Int64", "Unidades", "≥ 0; entregada ≤ solicitada; merma ≤ ingreso"],
    ["temperatura_cabina_c", "float64", "°C", "temperatura requerida del producto ± tolerancia; -999 es inválido"],
    ["temperatura_conservacion_requerida_c", "float64", "°C", "Observado en el maestro: -18 a 20 °C; debe estar informado"],
    ["tolerancia_temperatura_c", "float64", "°C", "Observado en el maestro: 2 a 5 °C; debe ser ≥ 0"],
    ["humedad_cabina_pct", "float64", "%", "0 a 100"],
    ["otif y banderas de desviación", "boolean / Int64", "0–1", "Valores permitidos: 0 y 1"],
    ["dias_en_almacen", "Int64", "Días", "≥ 0 y consistente con las fechas del movimiento"],
], columns=["variable", "tipo_esperado", "unidad", "regla_o_rango"])

display(diccionario_variables)

,variable,tipo_esperado,unidad,regla_o_rango
0,"order_id, producto_id, camion_id, lote_id",string / clave,Sin unidad,No nulo y formato normalizado; único según la ...
1,"fecha_despacho, timestamp",datetime,Fecha y hora,Fecha válida y coherente con la secuencia oper...
2,"fecha_ingreso, fecha_salida, fecha_vencimiento",date/datetime,Fecha,salida ≥ ingreso y vencimiento ≥ ingreso
3,"cantidades solicitada, entregada, ingreso, sal...",Int64,Unidades,≥ 0; entregada ≤ solicitada; merma ≤ ingreso
4,temperatura_cabina_c,float64,°C,temperatura requerida del producto ± toleranci...
5,temperatura_conservacion_requerida_c,float64,°C,Observado en el maestro: -18 a 20 °C; debe est...
6,tolerancia_temperatura_c,float64,°C,Observado en el maestro: 2 a 5 °C; debe ser ≥ 0
7,humedad_cabina_pct,float64,%,0 a 100
8,otif y banderas de desviación,boolean / Int64,0–1,Valores permitidos: 0 y 1
9,dias_en_almacen,Int64,Días,≥ 0 y consistente con las fechas del movimiento


## 12. Preparación inicial sin perder trazabilidad

Se crean copias de trabajo. Los valores originales permanecen en `datos_raw`. Las anomalías se marcan con banderas y no se eliminan automáticamente.

In [35]:
# Copia de pedidos
wms = wms_raw.copy()
for columna in ["order_id", "cliente_id", "producto_id", "camion_id", "chofer_id", "centro_distribucion"]:
    wms[columna] = normalizar_texto(wms[columna])

for columna in [
    "cantidad_solicitada", "cantidad_entregada",
    "tiempo_entrega_prometido_hrs", "tiempo_entrega_real_hrs",
    "otif_on_time", "otif_in_full", "otif",
]:
    wms[columna] = a_numero(wms[columna])

wms["fecha_despacho_dt"] = a_fecha(wms_raw["fecha_despacho"])
wms["duplicado_order_id_flag"] = wms["order_id"].duplicated(keep=False).astype("Int64")
wms["fecha_invalida_flag"] = wms["fecha_despacho_dt"].isna().astype("Int64")
wms["tiempo_negativo_flag"] = wms["tiempo_entrega_real_hrs"].lt(0).astype("Int64")

# Copia de telemetría
iot = iot_raw.copy()
for columna in ["viaje_id", "order_id", "camion_id", "producto_id"]:
    iot[columna] = normalizar_texto(iot[columna])

iot["timestamp_dt"] = a_fecha(iot_raw["timestamp"])
iot["temperatura_original"] = a_numero(iot_raw["temperatura_cabina_c"])
iot["temp_unit_original"] = normalizar_texto(iot_raw["temp_unit"])
iot["humedad_cabina_pct"] = a_numero(iot_raw["humedad_cabina_pct"])
iot["temperatura_error_flag"] = iot["temperatura_original"].eq(-999).astype("Int64")

iot["temperatura_cabina_c"] = np.select(
    [
        iot["temp_unit_original"].eq("C"),
        iot["temp_unit_original"].eq("F"),
        iot["temp_unit_original"].eq("K"),
    ],
    [
        iot["temperatura_original"],
        (iot["temperatura_original"] - 32) * 5 / 9,
        iot["temperatura_original"] - 273.15,
    ],
    default=np.nan,
)
iot.loc[iot["temperatura_error_flag"].eq(1), "temperatura_cabina_c"] = np.nan
iot["temp_unit"] = "C"
iot["humedad_fuera_rango_flag"] = (~iot["humedad_cabina_pct"].between(0, 100)).astype("Int64")

# Copia de productos
productos = productos_raw.copy()
productos["producto_id"] = normalizar_texto(productos["producto_id"])
productos["categoria_logistica"] = (
    productos["categoria_logistica"]
    .astype("string")
    .str.strip()
    .str.capitalize()
    .replace({"Conjelado": "Congelado"})
)
productos["temperatura_conservacion_requerida_c"] = a_numero(productos_raw["temperatura_conservacion_requerida_c"])
productos["tolerancia_temperatura_c"] = a_numero(productos_raw["tolerancia_temperatura_c"])

# Copia de inventario
inventario = inventario_raw.copy()
for columna in ["movimiento_id", "lote_id", "producto_id", "centro_distribucion"]:
    inventario[columna] = normalizar_texto(inventario[columna])

for columna in ["cantidad_ingreso", "cantidad_salida", "cantidad_merma", "dias_en_almacen", "costo_unitario_bob"]:
    inventario[columna] = a_numero(inventario_raw[columna])

for columna in ["fecha_ingreso", "fecha_salida", "fecha_vencimiento"]:
    inventario[columna + "_dt"] = a_fecha(inventario_raw[columna])

inventario["duplicado_movimiento_flag"] = inventario["movimiento_id"].duplicated(keep=False).astype("Int64")
inventario["merma_negativa_flag"] = inventario["cantidad_merma"].lt(0).astype("Int64")
inventario["salida_antes_ingreso_flag"] = (
    inventario["fecha_salida_dt"] < inventario["fecha_ingreso_dt"]
).astype("Int64")
inventario["vencimiento_antes_ingreso_flag"] = (
    inventario["fecha_vencimiento_dt"] < inventario["fecha_ingreso_dt"]
).astype("Int64")

print("Copias de trabajo creadas sin modificar los DataFrames Bronze originales.")

Copias de trabajo creadas sin modificar los DataFrames Bronze originales.


In [36]:
verificacion_preparacion = pd.DataFrame([
    ["Pedidos", len(wms), int(wms["duplicado_order_id_flag"].sum()), int(wms["fecha_invalida_flag"].sum())],
    ["Telemetría", len(iot), int(iot["temperatura_error_flag"].sum()), int(iot["humedad_fuera_rango_flag"].sum())],
    ["Inventario", len(inventario), int(inventario["duplicado_movimiento_flag"].sum()), int(inventario["merma_negativa_flag"].sum())],
], columns=["dataset", "filas_conservadas", "alerta_principal_1", "alerta_principal_2"])

display(verificacion_preparacion)

assert len(wms) == len(wms_raw), "Se perdieron filas de pedidos durante la preparación."
assert len(iot) == len(iot_raw), "Se perdieron filas de telemetría durante la preparación."
assert len(inventario) == len(inventario_raw), "Se perdieron filas de inventario durante la preparación."
assert set(iot["temp_unit"].dropna().unique()) == {"C"}, "La unidad de temperatura no quedó normalizada."

print("Validación superada: se conservaron todas las filas y la temperatura quedó expresada en °C.")

,dataset,filas_conservadas,alerta_principal_1,alerta_principal_2
0,Pedidos,7550,100,4
1,Telemetría,28920,120,15
2,Inventario,6040,80,6


Validación superada: se conservaron todas las filas y la temperatura quedó expresada en °C.


## 13. Conclusión para la ficha

La capa Bronze de AndinaLog presenta nulos, duplicados, identificadores inconsistentes, fechas imposibles, unidades de temperatura mezcladas, valores centinela y reglas cronológicas incumplidas. Estas alertas deben conservarse como evidencia y tratarse con reglas explícitas antes de integrar pedidos, inventario, productos y telemetría.

Para entregar el trabajo, ejecute **Entorno de ejecución → Ejecutar todas**, compruebe que no existan errores y guarde el notebook con sus resultados. Después comparta el archivo desde Colab o Drive con permiso de lectura y copie ese enlace en la ficha del dataset.